# 00 — Setup and train/validation/test split

Creates the **single** 60/20/20 split used by every later notebook and saves the row indices:

- `results/splits/clf_split_indices.csv` — Telco Churn, **stratified** on `Churn` (Q1, Q2, Q4, Q5)
- `results/splits/reg_split_indices.csv` — California Housing, random (Q3)

The split is made **before any transformation is fitted** (split first, then fit). If a split file already exists it is **not** recreated, only loaded and checked, so *Restart & Run All* never changes the split.

Only split sizes and the class ratio of each part are inspected here; the test set's contents are not looked at.

In [1]:
import sys
from pathlib import Path

# Make the project root importable whether the notebook runs from root or notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils.py").exists())
sys.path.insert(0, str(ROOT))

from src.utils import *

set_seed()
print("Project root:", ROOT)
print("SEED =", SEED)

Project root: C:\Users\berks\CENG463_Midterm
SEED = 463


## 1. Telco Churn — stratified 60/20/20

In [2]:
X, y, ids = load_telco()

if CLF_SPLIT_FILE.exists():
    print(f"[skip] {CLF_SPLIT_FILE.name} already exists - loading it, not re-splitting.")
else:
    make_split(y, CLF_SPLIT_FILE, stratify=True, ids=ids)
    print(f"[ok] created {CLF_SPLIT_FILE.name}")

clf = load_split(X, y, CLF_SPLIT_FILE, ids=ids)

[ok] created clf_split_indices.csv


## 2. California Housing — random 60/20/20

In [3]:
Xh, yh = load_housing()

if REG_SPLIT_FILE.exists():
    print(f"[skip] {REG_SPLIT_FILE.name} already exists - loading it, not re-splitting.")
else:
    make_split(yh, REG_SPLIT_FILE, stratify=False)
    print(f"[ok] created {REG_SPLIT_FILE.name}")

reg = load_split(Xh, yh, REG_SPLIT_FILE)

[ok] created reg_split_indices.csv


## 3. Integrity checks

Parts must be disjoint and together cover every row exactly once.

In [4]:
def check_split(parts, n_total, name):
    idx = {p: set(parts[f"X_{p}"].index) for p in ("train", "val", "test")}
    assert not (idx["train"] & idx["val"]), "train/val overlap"
    assert not (idx["train"] & idx["test"]), "train/test overlap"
    assert not (idx["val"] & idx["test"]), "val/test overlap"
    assert len(idx["train"] | idx["val"] | idx["test"]) == n_total, "rows missing"
    for p in ("train", "val", "test"):
        assert parts[f"X_{p}"].index.equals(parts[f"y_{p}"].index), f"X/y misaligned in {p}"
    print(f"[ok] {name}: disjoint, complete, aligned ({n_total} rows)")

check_split(clf, len(y), "Telco")
check_split(reg, len(yh), "California")

[ok] Telco: disjoint, complete, aligned (7043 rows)
[ok] California: disjoint, complete, aligned (20640 rows)


## 4. Exact split sizes (saved for the report)

`target_mean` = churn rate for Telco, mean `MedHouseVal` (×100,000 USD) for California.

In [5]:
clf_sizes = split_size_table(clf)
full_row = {"split": "full", "n": len(y), "fraction": 1.0, "target_mean": y.mean()}
clf_sizes = pd.concat([clf_sizes, pd.DataFrame([full_row])], ignore_index=True)
save_table(clf_sizes, "clf_split_sizes")
clf_sizes

,split,n,fraction,target_mean
0,train,4225,0.599886,0.265325
1,val,1409,0.200057,0.265436
2,test,1409,0.200057,0.265436
3,full,7043,1.000000,0.265370


In [6]:
reg_sizes = split_size_table(reg)
full_row = {"split": "full", "n": len(yh), "fraction": 1.0, "target_mean": yh.mean()}
reg_sizes = pd.concat([reg_sizes, pd.DataFrame([full_row])], ignore_index=True)
save_table(reg_sizes, "reg_split_sizes")
reg_sizes

,split,n,fraction,target_mean
0,train,12384,0.6,2.073573
1,val,4128,0.2,2.049857
2,test,4128,0.2,2.072216
3,full,20640,1.0,2.068558
